# 04 - Demoiselles prototyping

Build and check the face contours, seed points, bounded Voronoi
tessellation, face clipping, and the gender/decade assignment and final
render here, cell by cell, before any of it moves into `src/`. Nothing in
this notebook is wired into the rest of the project until Task 5
graduates the validated code.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import plotly.graph_objects as go
from shapely.geometry import Point, Polygon

from src import config

palette = config.PALETTES["demoiselles"]

## Face contours

Rough polygons digitized from `images/les_demoiselles_davignon.png`, one
per figure. These are never subdivided by seed points and never carry
data — they render as flat decorative fills.

In [ ]:
FACE_CONTOURS = [
    [(0.20, 0.70), (0.25, 0.70), (0.30, 0.80), (0.25, 0.90), (0.20, 0.85), (0.15, 0.80)],
    [(0.35, 0.65), (0.45, 0.65), (0.45, 0.80), (0.40, 0.85), (0.35, 0.80), (0.30, 0.70)],
    [(0.50, 0.75), (0.55, 0.70), (0.60, 0.80), (0.60, 0.90), (0.50, 0.95), (0.45, 0.85)],
    [(0.75, 0.75), (0.85, 0.80), (0.90, 0.90), (0.80, 0.95), (0.75, 0.90), (0.70, 0.80)],
    [(0.75, 0.45), (0.85, 0.45), (0.90, 0.60), (0.85, 0.65), (0.75, 0.65), (0.70, 0.55)],
]

FACE_POLYGONS = [Polygon(points) for points in FACE_CONTOURS]

for polygon in FACE_POLYGONS:
    assert polygon.is_valid
len(FACE_POLYGONS)

## Category counts (drives cell count and target area)

Every (decade, gender) category actually present in the cleaned data
gets its own cell, capped at `MAX_CELLS` (pooling isn't implemented for
the weighted version yet -- real data has 24 categories, well under the
cap). This must run before the anchors are placed, since the anchor
list below is ordered by category rank.

In [ ]:
def classify_gender(raw):
    if not isinstance(raw, str):
        return None
    g = raw.strip().lower()
    if "trans" in g:
        return "Transgender"
    if g.startswith("female"):
        return "Woman"
    if g.startswith("male"):
        return "Man"
    return None

assert classify_gender("male") == "Man"
assert classify_gender("female") == "Woman"
assert classify_gender("female (transwoman)") == "Transgender"
assert classify_gender("male (trans? ftm?)") == "Transgender"
assert classify_gender("transgender woman") == "Transgender"
assert classify_gender("") is None
assert classify_gender("non-binary") is None
assert classify_gender("gender non-conforming") is None
print("all gender classification checks passed")

In [ ]:
def person_gender_decade_counts(df):
    known = df[df["Decade_acquired"] != "unknown"]
    counts = {}
    for genders, decade in zip(known["Gender"], known["Decade_acquired"]):
        if not isinstance(genders, list):
            continue
        for raw in genders:
            bucket = classify_gender(raw)
            if bucket is None:
                continue
            key = (decade, bucket)
            counts[key] = counts.get(key, 0) + 1
    return counts

In [ ]:
from src import data

df = data.load_raw_data()
cleaned = data.clean_artworks(df)
counts = person_gender_decade_counts(cleaned)

MAX_CELLS = 40
n_cells = min(len(counts), MAX_CELLS)

# CATEGORY_ITEMS fixes the index <-> (decade, gender) mapping used by every
# later cell (ANCHORS, weights, final cells are all indexed the same way)
CATEGORY_ITEMS = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
len(counts), n_cells

## Anchor points (placed by hand)

Unlike the earlier jittered-grid version, anchor **position** is now
fixed by hand, the same workflow as `FACE_CONTOURS`: edit the
coordinates below, re-run this cell and the debug view after it, and
iterate until the layout reads well against the painting. The solver
(further down) only adjusts each anchor's **weight** to hit its target
area -- it never moves a point away from where you put it.

The list is ordered by `CATEGORY_ITEMS` rank (index 0 = most-acquired
category, `('1960s', 'Man')` with 44170 works). The starting values
below are just a jittered-grid bootstrap -- edit freely; the comments
tell you which category each row controls.

In [ ]:
def generate_seed_points(n, seed=config.RANDOM_STATE):
    """One-off bootstrap for the starting ANCHORS values below -- not
    used anywhere in the final pipeline once positions are hand-placed."""
    rng = np.random.default_rng(seed)
    grid_size = int(np.ceil(np.sqrt(n * 1.5)))
    xs = np.linspace(0.03, 0.97, grid_size)
    ys = np.linspace(0.03, 0.97, grid_size)
    candidates = [(x, y) for x in xs for y in ys]
    jitter = rng.uniform(-0.03, 0.03, size=(len(candidates), 2))
    jittered = [(x + jx, y + jy) for (x, y), (jx, jy) in zip(candidates, jitter)]
    points = [
        (x, y) for x, y in jittered
        if not any(polygon.contains(Point(x, y)) for polygon in FACE_POLYGONS)
    ]
    rng.shuffle(points)
    return points[:n]

# bootstrap_points = generate_seed_points(len(CATEGORY_ITEMS))
# for i, ((label, count), (x, y)) in enumerate(zip(CATEGORY_ITEMS, bootstrap_points)):
#     print(f"    ({x:.2f}, {y:.2f}),  # {i}: {label} ({count})")

In [ ]:
ANCHORS = [
    (0.42, 0.01),  # 0: ('1960s', 'Man') (44170)
    (0.42, 0.62),  # 1: ('2010s', 'Man') (20360)
    (0.79, 0.01),  # 2: ('2000s', 'Man') (20334)
    (0.76, 0.19),  # 3: ('1970s', 'Man') (11697)
    (0.40, 0.95),  # 4: ('1980s', 'Man') (9452)
    (0.94, 0.78),  # 5: ('1990s', 'Man') (9164)
    (0.01, 0.78),  # 6: ('2010s', 'Woman') (6911)
    (0.01, 0.43),  # 7: ('1940s', 'Man') (6733)
    (0.23, 0.05),  # 8: ('1950s', 'Man') (6023)
    (0.61, 0.80),  # 9: ('2000s', 'Woman') (5752)
    (0.76, 0.95),  # 10: ('2020s', 'Man') (5710)
    (0.22, 0.38),  # 11: ('1990s', 'Woman') (2935)
    (0.80, 0.42),  # 12: ('2020s', 'Woman') (2040)
    (0.97, 0.23),  # 13: ('1970s', 'Woman') (1664)
    (0.05, 0.03),  # 14: ('1930s', 'Man') (1655)
    (0.98, 0.41),  # 15: ('1980s', 'Woman') (1190)
    (0.25, 0.99),  # 16: ('1960s', 'Woman') (1156)
    (0.59, 0.43),  # 17: ('1940s', 'Woman') (657)
    (0.21, 0.20),  # 18: ('1950s', 'Woman') (390)
    (0.95, 0.96),  # 19: ('2020s', 'Transgender') (63)
    (0.61, 0.58),  # 20: ('1930s', 'Woman') (50)
    (0.39, 0.42),  # 21: ('1940s', 'Transgender') (10)
    (0.40, 0.19),  # 22: ('1920s', 'Man') (9)
    (0.05, 0.61),  # 23: ('1950s', 'Transgender') (1)
]
assert len(ANCHORS) == len(CATEGORY_ITEMS)
len(ANCHORS)

## Debug view: anchors + labels over the real painting

Re-run after every edit to `ANCHORS` above. Each point is labelled
`index: decade gender (count)` so you can tell which one to move.

In [ ]:
import base64

with open(config.IMAGES_DIR / "les_demoiselles_davignon.png", "rb") as f:
    encoded_image = base64.b64encode(f.read()).decode()

def contour_trace(points, color):
    closed = list(points) + [points[0]]
    xs, ys = zip(*closed)
    return go.Scatter(
        x=list(xs), y=list(ys), mode="lines",
        line=dict(color=color, width=2), fill="none",
        hoverinfo="skip", showlegend=False,
    )

def anchor_labels_trace(anchors, items, color):
    xs = [p[0] for p in anchors]
    ys = [p[1] for p in anchors]
    labels = [f"{i}: {d} {g[:3]} ({c})" for i, ((d, g), c) in enumerate(items)]
    return go.Scatter(
        x=xs, y=ys, mode="markers+text",
        marker=dict(size=6, color=color),
        text=labels, textposition="top center",
        textfont=dict(color=color, size=9),
        hoverinfo="skip", showlegend=False,
    )

fig = go.Figure()
for contour in FACE_CONTOURS:
    fig.add_trace(contour_trace(contour, "#FF00FF"))
fig.add_trace(anchor_labels_trace(ANCHORS, CATEGORY_ITEMS, "#00FFFF"))
fig.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="x", yref="y",
        x=0, y=1, sizex=1, sizey=1,
        xanchor="left", yanchor="top",
        sizing="stretch", layer="below",
    )
)
fig.update_layout(
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1], scaleanchor="x"),
    showlegend=False,
    margin=dict(t=20, l=0, r=0, b=0),
    height=800,
)
fig.show()

## Hand off: place the anchors

Adjust `ANCHORS` above (one line per category, ordered by count --
see the comments) and re-run it plus the debug view until the layout
reads well: big categories where you want visual weight, small ones
tucked into corners, none sitting inside a face contour. Do not
continue to the solver below until you're happy with the positions --
the solver never moves them, it only sizes them.

## Weighted Voronoi (power diagram, weight-only)

Same power-diagram mechanism as before (each site gets a weight `w_i`,
a point belongs to the site minimizing `|x - p_i|^2 - w_i`), but the
solver **no longer moves anchor positions** -- only weights are
adjusted, since positions are now placed by hand above. This also
simplifies the solver: no more competing between weight-driven area
and Lloyd-relaxation drift.

Target area is `sqrt(count)`-scaled rather than linear-in-count: with
real data this skewed (1 to 44,170 works), a linear-area/floor scheme
collapsed most of the tail into identically-sized cells (an 8-category
pileup at the same floor value). `sqrt` keeps every category's area
strictly ordered and distinct without needing an artificial floor.

In [ ]:
total_weight = sum(count for _, count in CATEGORY_ITEMS)
sqrt_weights = np.array([count ** 0.5 for _, count in CATEGORY_ITEMS])
TARGET_FRACS = sqrt_weights / sqrt_weights.sum()

TARGET_FRACS.min(), TARGET_FRACS.max(), TARGET_FRACS.sum()

In [ ]:
from scipy.spatial import ConvexHull
from shapely.geometry import MultiPoint, box
from collections import defaultdict

def _power_center(p1, w1, p2, w2, p3, w3):
    """The point equidistant, in power-distance, from all 3 weighted
    sites -- the power-diagram analogue of a circumcenter. Solves a 2x2
    linear system derived by subtracting the power-distance equations
    pairwise (the quadratic terms cancel)."""
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    A = np.array([[2 * (x2 - x1), 2 * (y2 - y1)], [2 * (x3 - x1), 2 * (y3 - y1)]])
    b = np.array([
        (w1 - w2) + (x2**2 - x1**2) + (y2**2 - y1**2),
        (w1 - w3) + (x3**2 - x1**2) + (y3**2 - y1**2),
    ])
    if abs(np.linalg.det(A)) < 1e-12:
        return None
    return tuple(np.linalg.solve(A, b))

def _mirror_weighted(points, weights, bounds):
    minx, miny, maxx, maxy = bounds
    mirrored_points, mirrored_weights = [], []
    for (x, y), w in zip(points, weights):
        mirrored_points += [(2 * minx - x, y), (2 * maxx - x, y), (x, 2 * miny - y), (x, 2 * maxy - y)]
        mirrored_weights += [w, w, w, w]
    return mirrored_points, mirrored_weights

def power_diagram(points, weights, bounds=(0.0, 0.0, 1.0, 1.0)):
    """Power/Laguerre-Voronoi diagram of weighted sites, closed inside
    bounds via the same mirror-across-every-edge trick used for the
    ordinary Voronoi diagram. Returns {site_index: Polygon}, omitting
    any site whose cell vanished (fewer than 3 power-center vertices)."""
    n = len(points)
    mirror_points, mirror_weights = _mirror_weighted(points, weights, bounds)
    all_points = list(points) + mirror_points
    all_weights = list(weights) + mirror_weights
    lifted = np.array([[x, y, x * x + y * y - w] for (x, y), w in zip(all_points, all_weights)])
    hull = ConvexHull(lifted)
    lower_hull = hull.simplices[hull.equations[:, 2] < -1e-9]

    site_vertices = defaultdict(list)
    for i, j, k in lower_hull:
        center = _power_center(
            all_points[i], all_weights[i], all_points[j], all_weights[j], all_points[k], all_weights[k]
        )
        if center is None:
            continue
        for site in (i, j, k):
            site_vertices[site].append(center)

    boundary = box(*bounds)
    cells = {}
    for i in range(n):
        vertices = site_vertices.get(i)
        if not vertices or len(vertices) < 3:
            continue
        hull_polygon = MultiPoint(vertices).convex_hull
        if hull_polygon.geom_type != "Polygon":
            continue
        cell = hull_polygon.intersection(boundary)
        if not cell.is_empty and cell.geom_type == "Polygon":
            cells[i] = cell
    return cells

In [ ]:
def solve_cell_weights(anchors, target_fracs, bounds=(0.0, 0.0, 1.0, 1.0), iterations=250, weight_lr=0.4):
    """Iteratively adjusts each site's weight to push its cell area
    toward target_fracs. Positions are never touched -- anchors are
    placed by hand, so the solver's only job is sizing."""
    n = len(anchors)
    weights = np.zeros(n)
    total_area = (bounds[2] - bounds[0]) * (bounds[3] - bounds[1])

    for _ in range(iterations):
        cells = power_diagram(anchors, weights, bounds)
        areas = np.array([cells[i].area if i in cells else 0.0 for i in range(n)])
        error = target_fracs - areas / total_area
        weights = weights + weight_lr * error
        weights -= weights.mean()  # power diagrams are invariant to a global weight shift

    return power_diagram(anchors, weights, bounds), weights

WEIGHTED_CELLS, _SITE_WEIGHTS = solve_cell_weights(ANCHORS, TARGET_FRACS)
len(WEIGHTED_CELLS)

## Face clipping and convergence check

Same face-clipping rule as before: a cell is intersected against any
face it overlaps, subtracting the overlap so faces stay undivided.
Applied *after* the solver converges, not during.

In [ ]:
def _clip_faces_one(cell, face_polygons):
    result = cell
    for face in face_polygons:
        if result.intersects(face):
            result = result.difference(face)
    return result

DEMOISELLES_CELLS = {}
for i, cell in WEIGHTED_CELLS.items():
    clipped = _clip_faces_one(cell, FACE_POLYGONS)
    if not clipped.is_empty:
        DEMOISELLES_CELLS[i] = clipped

mean_abs_error = np.mean([
    abs(TARGET_FRACS[i] - DEMOISELLES_CELLS[i].area) if i in DEMOISELLES_CELLS else TARGET_FRACS[i]
    for i in range(len(CATEGORY_ITEMS))
])
len(DEMOISELLES_CELLS), len(CATEGORY_ITEMS), mean_abs_error

In [ ]:
def polygon_to_traces(polygon, fillcolor, line_color, hovertext=None):
    geoms = polygon.geoms if polygon.geom_type == "MultiPolygon" else [polygon]
    traces = []
    for geom in geoms:
        xs, ys = geom.exterior.xy
        traces.append(go.Scatter(
            x=list(xs), y=list(ys),
            fill="toself", fillcolor=fillcolor,
            line=dict(color=line_color, width=2),
            mode="lines", hoveron="fills", name="",
            text=hovertext, hoverinfo="text" if hovertext else "skip",
            showlegend=False,
        ))
    return traces

# uncolored preview: just the shapes and relative sizes
fig = go.Figure()
for cell in DEMOISELLES_CELLS.values():
    for trace in polygon_to_traces(cell, palette["background"], palette["black"]):
        fig.add_trace(trace)
fig.update_layout(
    plot_bgcolor=palette["background"],
    xaxis=dict(visible=False, range=[0, 1]),
    yaxis=dict(visible=False, range=[0, 1], scaleanchor="x"),
    showlegend=False,
    margin=dict(t=20, l=0, r=0, b=0),
)
fig.show()

## Faces as real image crops

Instead of a flat decorative fill, each face polygon now shows the
actual painting pixels in that region: a PIL mask shaped like the
polygon clips the painting to an RGBA cutout, cropped to the
polygon's bounding box, then placed with `fig.add_layout_image` at
that exact box in plot coordinates. Plotly can't clip an image to an
arbitrary polygon natively, so the clipping happens in PIL first and
the result is just a positioned rectangle with transparent corners.

In [ ]:
from io import BytesIO
from PIL import Image, ImageDraw

_PAINTING = Image.open(config.IMAGES_DIR / "les_demoiselles_davignon.png").convert("RGB")
_PAINTING_W, _PAINTING_H = _PAINTING.size

def face_image_overlay(contour):
    """Masks _PAINTING to contour's shape (plot-space, y=0 at bottom)
    and returns a layout_image dict positioned at the polygon's bounding
    box in plot coordinates."""
    px_points = [(x * _PAINTING_W, (1 - y) * _PAINTING_H) for x, y in contour]
    xs = [p[0] for p in px_points]
    ys = [p[1] for p in px_points]
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(ys), max(ys)

    mask = Image.new("L", (_PAINTING_W, _PAINTING_H), 0)
    ImageDraw.Draw(mask).polygon(px_points, fill=255)
    rgba = _PAINTING.convert("RGBA")
    rgba.putalpha(mask)
    crop = rgba.crop((int(x0), int(y0), int(x1) + 1, int(y1) + 1))

    buf = BytesIO()
    crop.save(buf, format="PNG")
    encoded = base64.b64encode(buf.getvalue()).decode()

    return dict(
        source=f"data:image/png;base64,{encoded}",
        xref="x", yref="y",
        x=x0 / _PAINTING_W, y=1 - y0 / _PAINTING_H,
        sizex=(x1 - x0) / _PAINTING_W, sizey=(y1 - y0) / _PAINTING_H,
        xanchor="left", yanchor="top", sizing="stretch", layer="above",
    )

FACE_IMAGE_OVERLAYS = [face_image_overlay(contour) for contour in FACE_CONTOURS]
len(FACE_IMAGE_OVERLAYS)

## Final render, with real data

Plotly doesn't auto-generate a legend for `fill='toself'` traces, so
three invisible proxy marker traces (one per gender bucket) are added
purely to produce the three legend entries.

In [ ]:
_GENDER_PALETTE_KEYS = {"Woman": "woman", "Man": "man", "Transgender": "transgender"}

def _legend_proxy_traces(palette):
    return [
        go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=10, color=palette[palette_key]),
            name=label, showlegend=True,
        )
        for label, palette_key in _GENDER_PALETTE_KEYS.items()
    ]

In [ ]:
def demoiselles_voronoi(df):
    df_counts = person_gender_decade_counts(df)
    df_items = sorted(df_counts.items(), key=lambda kv: kv[1], reverse=True)

    fig = go.Figure()

    for i, cell in DEMOISELLES_CELLS.items():
        if i >= len(df_items):
            continue
        (decade, gender), count = df_items[i]
        fillcolor = palette[_GENDER_PALETTE_KEYS[gender]]
        hovertext = f"{decade}<br>{count} artworks"
        for trace in polygon_to_traces(cell, fillcolor, palette["black"], hovertext=hovertext):
            fig.add_trace(trace)

    for trace in _legend_proxy_traces(palette):
        fig.add_trace(trace)

    for overlay in FACE_IMAGE_OVERLAYS:
        fig.add_layout_image(overlay)

    fig.update_layout(
        plot_bgcolor=palette["background"],
        xaxis=dict(visible=False, range=[0, 1]),
        yaxis=dict(visible=False, range=[0, 1], scaleanchor="x"),
        showlegend=True,
        margin=dict(t=20, l=0, r=0, b=0),
    )
    return fig

demoiselles_voronoi(cleaned).show()